# value_head_v14 — train on pillar3b backbone + A/B verify + Q-sweep

Pillar3b is the new policy SOA. Per HISTORY 138, value head must be retrained when the backbone moves. This notebook produces `value_head_v14.pt`, A/B verifies against `value_head_sharp25_ep12.pt` (pillar3a's head, which is mis-paired to pillar3b's backbone), and runs the q-sweep with the matched head.

## Files needed on Drive (`MyDrive/alphatrain/`)

| file | size | source | purpose |
|---|---|---|---|
| `colorlines_path_b.tar.gz` | ~1.2 MB | local: `colorlines_path_b.tar.gz` | code (incl. `train_value_head.py`) |
| `pillar3b_epoch_20.pt` | ~136 MB | local: `alphatrain/data/pillar3b_epoch_20.pt` | backbone for value head + policy for eval |
| `value_targets_v13.pt.gz` | TBD | **build locally on M5 from V13 JSONs** — see cell 2 instructions | train data: multi-horizon survival labels |
| `value_val_K64.pt` | ~59 KB | local: `alphatrain/data/value_val_K64.pt` | val data: K=64 rollout soft-labels |
| `value_head_sharp25_ep12.pt` | ~38 KB | local: `alphatrain/data/value_head_sharp25_ep12.pt` | reference (old) head for A/B |

### How to build `value_targets_v13.pt` on M5 (if not already done)

```bash
python -m alphatrain.scripts.build_value_targets \
    --games-dir data/selfplay_v13 data/crisis_v13 \
    --output alphatrain/data/value_targets_v13.pt
gzip -k alphatrain/data/value_targets_v13.pt   # → ~200-300 MB compressed
```

Then upload `value_targets_v13.pt.gz` to Drive.

If you'd rather use the existing `value_targets_v11.pt` (matched pillar3a's recipe), swap the path in cell 2 — both work as inputs to `train_value_head.py`.

## Steps

1. **Retrain** `value_head_v14.pt` on pillar3b + V13 survival targets (5 epochs, ~30 min on L4).
2. **A/B verify** on 100 seeds MCTS@100 cap=20K q=2.0. Gate per HISTORY 138.
3. **Q-sweep** q ∈ {1.5, 2.5, 3.0} with the matched head (~3-5h).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# 1) Code
!cp {DRIVE}/colorlines_path_b.tar.gz /content/
!cd /content && tar xzf colorlines_path_b.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)

# 2) Backbone + reference value head
shutil.copy(f'{DRIVE}/pillar3b_epoch_20.pt',
            '/content/alphatrain/data/pillar3b_epoch_20.pt')
shutil.copy(f'{DRIVE}/value_head_sharp25_ep12.pt',
            '/content/alphatrain/data/value_head_sharp25_ep12.pt')
shutil.copy(f'{DRIVE}/value_val_K64.pt',
            '/content/alphatrain/data/value_val_K64.pt')
print(f'Backbone: {os.path.getsize("/content/alphatrain/data/pillar3b_epoch_20.pt")/1e6:.0f} MB')
print(f'Reference head: {os.path.getsize("/content/alphatrain/data/value_head_sharp25_ep12.pt")/1e3:.0f} KB')
print(f'Val data: {os.path.getsize("/content/alphatrain/data/value_val_K64.pt")/1e3:.0f} KB')

# 3) Train-data tensor — copy to /content first, verify, then decompress.
# Streaming directly from Drive FUSE can truncate large files silently.
TRAIN_DATA_GZ_DRIVE = f'{DRIVE}/value_targets_v13.pt.gz'
TRAIN_DATA_PT_LOCAL = '/content/alphatrain/data/value_targets_v13.pt'
t0 = time.time()
!cp {TRAIN_DATA_GZ_DRIVE} /content/value_targets_v13.pt.gz
!gunzip -t /content/value_targets_v13.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/value_targets_v13.pt.gz > {TRAIN_DATA_PT_LOCAL}
pt_size = os.path.getsize(TRAIN_DATA_PT_LOCAL)
print(f'value_targets_v13.pt: {pt_size/1e9:.2f} GB ({time.time()-t0:.0f}s)')
!rm /content/value_targets_v13.pt.gz

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

## Step 1 — Train value_head_v14 (~30-60 min on L4)

Args follow `train_value_head.py`'s signature: `--train-data`, `--val-data`, `--out`. Same hyperparameters as pillar3a's value_head retrain (task #103, HISTORY 158): 5 epochs, batch 4096, lr 1e-3.

In [ ]:
%cd /content
!python -m alphatrain.scripts.train_value_head \
    --backbone alphatrain/data/pillar3b_epoch_20.pt \
    --train-data alphatrain/data/value_targets_v13.pt \
    --val-data alphatrain/data/value_val_K64.pt \
    --out alphatrain/data/value_head_v14.pt \
    --epochs 5 --batch-size 4096 --lr 1e-3

In [ ]:
# Backup to Drive immediately
import shutil, os
shutil.copy('/content/alphatrain/data/value_head_v14.pt',
            f'{DRIVE}/value_head_v14.pt')
print(f'Backed up {os.path.getsize(f"{DRIVE}/value_head_v14.pt")/1e3:.1f} KB')

## Step 2 — A/B verify (100 seeds, MCTS@100, cap=20K, q=2.0)

Decision gate per HISTORY 138:
- NEW head ≥ +5% over OLD on mean OR P50 → proceed with sweep
- Within ±3% → default to new (matched-by-definition)
- NEW < OLD by >5% → retraining failed; debug before sweep

Both runs use the same 100 seeds (0..99) for apples-to-apples.

In [ ]:
%cd /content
import os
os.makedirs('/content/qsweep_logs', exist_ok=True)

OLD_HEAD_LOG = '/content/qsweep_logs/ab_old.log'
print('=== OLD head (value_head_sharp25_ep12, mis-paired) ===')
!python -m alphatrain.scripts.eval_parallel \
    --model alphatrain/data/pillar3b_epoch_20.pt \
    --value-head-path alphatrain/data/value_head_sharp25_ep12.pt \
    --simulations 100 --top-k 30 --batch-size 8 \
    --device cuda --workers 16 --max-turns 20000 \
    --q-weight 2.0 --early-stop --compile \
    --seeds $(seq 0 99 | tr '\n' ' ') 2>&1 | tee {OLD_HEAD_LOG}
!cp {OLD_HEAD_LOG} {DRIVE}/qsweep_pillar3b_v14_ab_old.log

In [ ]:
%cd /content
import os
os.makedirs('/content/qsweep_logs', exist_ok=True)
print('=== NEW head (value_head_v14) — also serves as q=2.0 sweep point ===')
NEW_HEAD_LOG = '/content/qsweep_logs/q2.0.log'
!python -m alphatrain.scripts.eval_parallel \
    --model alphatrain/data/pillar3b_epoch_20.pt \
    --value-head-path alphatrain/data/value_head_v14.pt \
    --simulations 100 --top-k 30 --batch-size 8 \
    --device cuda --workers 16 --max-turns 20000 \
    --q-weight 2.0 --early-stop --compile \
    --seeds $(seq 0 99 | tr '\n' ' ') 2>&1 | tee {NEW_HEAD_LOG}
!cp {NEW_HEAD_LOG} {DRIVE}/qsweep_pillar3b_v14_q2.0.log

## Step 3 — Q-weight sweep with the new head (~3-5h)

If A/B passed, sweep q ∈ {**1.5, 2.5, 3.0**} × 100 seeds × MCTS@100 × cap=20K (q=2.0 already done in A/B step above — reused). Each q is ~1-2h on L4. Run sequentially in this notebook, or split into 3 parallel notebook tabs by editing the Q value.

Floor + low percentiles matter more than mean for this iteration.

In [ ]:
%cd /content
import os
os.makedirs('/content/qsweep_logs', exist_ok=True)

# q=2.0 was already produced by the A/B step (NEW head). Skip it here.
for Q in [1.5, 2.5, 3.0]:
    log = f'/content/qsweep_logs/q{Q}.log'
    drive_log = f'{DRIVE}/qsweep_pillar3b_v14_q{Q}.log'
    print(f'\n\n========== q={Q} (log: {log}) ==========')
    !python -m alphatrain.scripts.eval_parallel \
        --model alphatrain/data/pillar3b_epoch_20.pt \
        --value-head-path alphatrain/data/value_head_v14.pt \
        --simulations 100 --top-k 30 --batch-size 8 \
        --device cuda --workers 16 --max-turns 20000 \
        --q-weight {Q} --early-stop --compile \
        --seeds $(seq 0 99 | tr '\n' ' ') 2>&1 | tee {log}
    !cp {log} {drive_log}
    print(f'Saved {drive_log}')

In [ ]:
# Summary table — extract MCTS mean/P50/floor from each log
import re, os, glob

print(f"{'q':>5} | {'MCTS mean':>10} | {'P25':>5} | {'P50':>5} | {'P75':>5} | {'<1000':>6} | {'>10K':>5}")
print('-' * 60)
for log_path in sorted(glob.glob('/content/qsweep_logs/q*.log')):
    q = log_path.split('q')[-1].replace('.log', '')
    with open(log_path) as f:
        text = f.read()
    m_mean = re.search(r'^\s*MEAN\s*\|\s*\d+\s*\|\s*(\d+)', text, re.M)
    mcts_idx = text.find('MCTS percentiles')
    mcts_section = text[mcts_idx:] if mcts_idx >= 0 else ''
    m_p25 = re.search(r'P25=(\d+)', mcts_section)
    m_p50 = re.search(r'P50=(\d+)', mcts_section)
    m_p75 = re.search(r'P75=(\d+)', mcts_section)
    m_floor = re.search(r'<1000:\s*\d+\s*\(([\d.]+)%\)', mcts_section)
    m_above_10k = re.search(r'>10000:\s*\d+\s*\((\d+)%\)', mcts_section)
    mean = m_mean.group(1) if m_mean else '?'
    p25 = m_p25.group(1) if m_p25 else '?'
    p50 = m_p50.group(1) if m_p50 else '?'
    p75 = m_p75.group(1) if m_p75 else '?'
    floor = m_floor.group(1) + '%' if m_floor else '?'
    above = m_above_10k.group(1) + '%' if m_above_10k else '?'
    print(f'{q:>5} | {mean:>10} | {p25:>5} | {p50:>5} | {p75:>5} | {floor:>6} | {above:>5}')